# Shared model helpers

Definitions only. Model notebooks load this with `%run 06_model_helpers.ipynb`. Fold assignments are **read from disk**, never recreated here. The same probability and metric rules apply to every model.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

contract = json.loads(Path("outputs/folds/contract.json").read_text())
seed = contract["seed"]
feature_columns = contract["feature_columns"]


def load_model_data():
    data = pd.read_parquet("outputs/features/features.parquet")
    data = data[data["label"].notna()].reset_index(drop=True)
    data["label"] = data["label"].astype("int8")
    nearest = data["distance"] - data["distance_above_nearest_m"]
    data["distance_ratio"] = ((data["distance"] + 1) / (nearest + 1)).astype("float32")
    folds = pd.read_csv(contract["fold_file"], dtype={"trip_id": str})
    data["fold"] = data["group_id"].map(folds.set_index("group_id")["fold"])
    assert data["fold"].notna().all() and data["group_id"].is_monotonic_increasing
    assert data.groupby("trip_id")["fold"].nunique().eq(1).all()
    points = pd.read_csv("outputs/candidates/points.csv", dtype={"trip_id": str})
    points = points[~points["bad_gps"]].merge(folds[["group_id", "fold"]], on="group_id", validate="one_to_one")
    return data, points


def point_softmax(scores, group_id):
    scores = np.asarray(scores, dtype=float)
    assert np.isfinite(scores).all()
    groups = np.asarray(group_id)
    shifted = scores - pd.Series(scores).groupby(groups, sort=False).transform("max").to_numpy()
    weights = np.exp(shifted)
    totals = pd.Series(weights).groupby(groups, sort=False).transform("sum").to_numpy()
    return weights / totals


def normalize_point_proba(probabilities, group_id):
    values = np.asarray(probabilities, dtype=float)
    assert np.isfinite(values).all() and (values >= 0).all()
    groups = np.asarray(group_id)
    totals = pd.Series(values).groupby(groups, sort=False).transform("sum").to_numpy()
    sizes = pd.Series(values).groupby(groups, sort=False).transform("size").to_numpy()
    return np.divide(values, totals, out=1.0 / sizes, where=totals > 0)


def score_points(data, probabilities, points):
    rows = data[["group_id", "segment_id", "label"]].reset_index(drop=True).copy()
    rows["score"] = np.asarray(probabilities)
    assert rows["label"].notna().all() and np.isfinite(rows["score"]).all()
    np.testing.assert_allclose(rows.groupby("group_id")["score"].sum(), 1, atol=1e-6)
    rows = rows.sort_values(["group_id", "segment_id"]).reset_index(drop=True)
    winners = rows.groupby("group_id", sort=False)["score"].idxmax().to_numpy()
    predictions = np.zeros(len(rows), dtype=np.int8)
    predictions[winners] = 1
    labels = rows["label"].to_numpy(dtype=np.int8)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, labels=[0, 1], zero_division=0)
    positives = rows.loc[rows["label"] == 1, "score"].to_numpy()
    assert rows.groupby("group_id")["label"].sum().le(1).all()
    assert set(rows["group_id"].unique()) <= set(points["group_id"])
    return {
        "top1": float(rows.iloc[winners]["label"].sum() / len(points)),
        "macro_f1": float(f1.mean()), "precision": float(precision[1]), "recall": float(recall[1]),
        "log_loss": float(-np.log(np.clip(positives, 1e-15, 1)).mean()) if len(positives) else None,
        "candidate_coverage": len(positives) / len(points), "points": len(points),
        "candidate_misses": len(points) - len(positives),
        "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
    }

`distance_ratio` uses a 1-metre offset in both numerator and denominator to avoid division by zero; the nearest candidate has ratio 1. It is derived from geometry only.

Softmax is a normalization, not a calibration claim. Zero-sum classifier probabilities fall back to a uniform distribution within that point. Metric ties choose the lowest segment ID. Log loss is conditional on candidate coverage; Top-1 includes all valid points, even empty shortlists.